In [1]:
from tqdm import tqdm
from lightfm import LightFM
from lightfm.cross_validation import random_train_test_split
from lightfm.evaluation import precision_at_k
import json
import torch
import torch.nn as nn
import torch.optim as optim
from matplotlib import pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
import optuna
from catboost import CatBoostClassifier, Pool
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
import os
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

/home/egor/miniconda3/envs/torch_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RESULT_PREPROCESSED_PATH = r'/mnt/c/Users/Egor/Desktop/study/diploma/project/dataset_preprocessed'
#RESULT_PREPROCESSED_PATH = r'C:\Users\Egor\Desktop\study\diploma\project\dataset_preprocessed'

class CFG:
    optimize_lightfm = True
    train_lightfm = False
    optimize_two_tower = False
    train_tow_tower = False
    optimize_catboost = False
    train_catboost = False
    seed = 42

In [ ]:
def reduce_mem_usage(df):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024 ** 2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024 ** 2
    print(f'Memory: {start_mem:.2f} Mb -> {end_mem:.2f} Mb ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    return df


def load_data(data_dir):
    print("Загрузка данных...")
    transactions = reduce_mem_usage(pd.read_csv(os.path.join(data_dir, "transactions.csv")))
    articles = reduce_mem_usage(pd.read_csv(os.path.join(data_dir, "articles.csv")))
    customers = reduce_mem_usage(pd.read_csv(os.path.join(data_dir, "customers.csv")))
    transactions = transactions[transactions['t_dat'] > '2020-08-21']
    return transactions, articles, customers


def create_save_best_params_callback(filename):
    def save_best_params_callback(study, trial):
        if study.best_trial.number == trial.number:
            with open(filename, "w") as f:
                json.dump(study.best_params, f, indent=4)
            print(f"Trial {trial.number} finished. New best params saved.")
    return save_best_params_callback

In [4]:
transactions, articles, customers = load_data(RESULT_PREPROCESSED_PATH)

Загрузка данных...
Memory: 2637.23 Mb -> 879.08 Mb (66.7% reduction)


/tmp/ipykernel_5091/314821450.py:19: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_5091/314821450.py:19: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:


Memory: 44.29 Mb -> 19.53 Mb (55.9% reduction)
Memory: 303.55 Mb -> 150.47 Mb (50.4% reduction)


### Подбор гиперпараметров и обучение для Lightfm

In [5]:
from lightfm.data import Dataset
def get_lightfm_dataset(transactions,customers, articles):
    user_cat_cols = ['age_group', 'club_member_status', 'fashion_news_frequency', 'recency_cat', 'frequency_cat', 'unique_articles_cat', 'fav_product_group', 'Active', 'FN']
    customers = customers.copy()
    customers['Active'] = customers['Active'].fillna(0).astype(int).astype(str).apply(lambda x: f"Active_{x}")
    customers['FN'] = customers['FN'].fillna(0).astype(int).astype(str).apply(lambda x: f"FN_{x}")
    
    item_cat_cols = ['product_group_name', 'colour_group_name']
    transactions = transactions[transactions['t_dat'] > '2020-08-21']
    transactions = transactions.groupby(['customer_id','article_id']).agg({'price':'sum','t_dat':'count'}).reset_index()
    transactions = transactions[['customer_id','article_id','price']]
    transactions['price'] = transactions['price'].astype('category')
    transactions['price'] = transactions['price'].cat.codes
    
    all_user_ids = customers['customer_id'].unique()
    all_article_ids = articles['article_id'].unique()
    
    
    # Все возможные значения фичей пользователя
    user_feature_names = []
    for col in user_cat_cols:
        user_feature_names.extend([f"{col}_{v}" for v in customers[col].unique()])
    
    # Все возможные значения фичей товаров
    item_feature_names = []
    for col in item_cat_cols:
        item_feature_names.extend([f"{col}_{v}" for v in articles[col].unique()])
    
    dataset = Dataset()
    dataset.fit(users=all_user_ids,
                items=all_article_ids,
                user_features=user_feature_names,
                item_features=item_feature_names)
    
    def iter_interactions(df_transactions):
        for row in df_transactions.itertuples():
            yield row.customer_id, row.article_id, row.price
    
    interactions_matrix, weights_matrix  = dataset.build_interactions(iter_interactions(transactions))
    def build_user_features(dataset, customers_df, cat_cols):
        features = []
        for _, row in customers_df.iterrows():
            uid = row['customer_id']
            feat_vals = [f"{col}_{row[col]}" for col in cat_cols if pd.notna(row[col])]
            features.append((uid, feat_vals))
        return dataset.build_user_features(features, normalize=False)
    
    # Матрица признаков товаров
    def build_item_features(dataset, articles_df, cat_cols):
        features = []
        for _, row in articles_df.iterrows():
            aid = row['article_id']
            feat_vals = [f"{col}_{row[col]}" for col in cat_cols if pd.notna(row[col])]
            features.append((aid, feat_vals))
        return dataset.build_item_features(features, normalize=False)
    
    user_feature_matrix = build_user_features(dataset, customers, user_cat_cols)
    item_feature_matrix = build_item_features(dataset, articles, item_cat_cols)
    train_interactions, test_interactions  = random_train_test_split(interactions_matrix, test_percentage=0.2, random_state=42)
    return user_feature_matrix,  item_feature_matrix, train_interactions, test_interactions, interactions_matrix, weights_matrix


In [6]:
if CFG.optimize_lightfm:
    user_feature_matrix,  item_feature_matrix, train_interactions, test_interactions, interactions_matrix, weights_matrix = get_lightfm_dataset(transactions, customers, articles)
    def objective(trial):
        learning_rate = trial.suggest_float('learning_rate', 1e-4, 0.05, log=True)
        no_components = trial.suggest_int('no_components', 16, 150)
        loss = trial.suggest_categorical('loss', choices=["bpr", "warp", "logistic"])
        item_alpha = trial.suggest_float('item_alpha', 1e-9, 1e-3, log=True)
        user_alpha = trial.suggest_float('user_alpha', 1e-9, 1e-3, log=True)
        max_sampled = trial.suggest_int('max_sampled', 5, 80)
        learning_schedule = trial.suggest_categorical('learning_schedule', choices=["adagrad", "adadelta"])
    
        model = LightFM(no_components=no_components,
                        learning_rate=learning_rate,
                        loss=loss,
                        item_alpha=item_alpha,
                        user_alpha=user_alpha,
                        max_sampled=max_sampled,
                        learning_schedule=learning_schedule,
                        random_state=42)
    
        model.fit(interactions_matrix,
                  sample_weight=weights_matrix,
                  user_features=user_feature_matrix,
                  item_features=item_feature_matrix,
                  epochs=20,
                  num_threads=20,
                  verbose=True)
        prec = precision_at_k(model=model,
                              train_interactions=train_interactions,
                              test_interactions=test_interactions,
                              user_features=user_feature_matrix,
                              item_features=item_feature_matrix,
                              num_threads=22,
                              k=12,
                              check_intersections=True).mean()
        return prec

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=25, show_progress_bar=True, callbacks=[create_save_best_params_callback('best_params_lightfm.json')])
    print("Лучшие гиперпараметры:", study.best_params)
    print("Лучшее Precision@12:", study.best_value)


In [7]:
if CFG.train_lightfm:
    calculate_map_for_every_epoch = False # too much time required only for graphic
    user_feature_matrix, item_feature_matrix, train_interactions, test_interactions, interactions_matrix, weights_matrix = get_lightfm_dataset(transactions, customers, articles)
    with open('best_params_lightfm.json', 'r') as file:
        best_params = json.load(file)
    map12_scores = []
    
    epochs = 40
    num_threads = 22
    final_model = LightFM(**best_params)
    for epoch in range(epochs):
        final_model.fit_partial(
            interactions_matrix,
            sample_weight=weights_matrix,
            user_features=user_feature_matrix,
            item_features=item_feature_matrix,
            epochs=1,
            num_threads=num_threads,
            verbose=True
        )
        if calculate_map_for_every_epoch:
            map12 = precision_at_k(
                final_model,
                train_interactions,
                test_interactions,
                user_features=user_feature_matrix,
                item_features=item_feature_matrix,
                k=12,
                num_threads=22,
                check_intersections=True
            ).mean()
            map12_scores.append(map12)
            print(f'MAP@12 [{map12}]')
        
        print(f"Epoch {epoch+1:2d}") 
    if calculate_map_for_every_epoch:
        last_map12 = map12_scores[-1]
    else:
        last_map12 = precision_at_k(
                final_model,
                train_interactions,
                test_interactions,
                user_features=user_feature_matrix,
                item_features=item_feature_matrix,
                k=12,
                num_threads=22,
                check_intersections=True
            ).mean()
        
    print(f'Last map12 [{last_map12}]')
    if calculate_map_for_every_epoch:
        epochs = np.arange(1, 41)
        fig, ax1 = plt.subplots(1, 1, figsize=(10, 8))
        
        ax1.plot(epochs, map12_scores, 'r-', label='MAP@12')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('MAP@12')
        ax1.set_title('Mean Average Precision @12 on Test Subset')
        ax1.grid(True)
        ax1.legend()
        
        plt.tight_layout()
        plt.savefig('training_curve_lightfm.png', dpi=150)
        plt.show()
    

### Подбор гиперпараметров и обучение для TwoTower

In [ ]:
from torch.utils.data import Dataset, DataLoader
class HMDataset(Dataset):
    def __init__(self, df, user_features_dict, item_features_dict, user_cat_cols, item_cat_cols, user_id_to_idx, item_id_to_idx):

        self.user_idx = df['user_idx'].values
        self.item_idx = df['item_idx'].values
        self.weights = df['weight'].values.astype(np.float32)

        self.user_features = []
        self.item_features = []

        idx_to_user = {v: k for k, v in user_id_to_idx.items()}
        idx_to_item = {v: k for k, v in item_id_to_idx.items()}

        for ui, ii in zip(self.user_idx, self.item_idx):
            uid = idx_to_user[ui]
            iid = idx_to_item[ii]
            self.user_features.append(user_features_dict.get(uid, [0]*len(user_cat_cols)))
            self.item_features.append(item_features_dict.get(iid, [0]*len(item_cat_cols)))

    def __len__(self):
        return len(self.user_idx)

    def __getitem__(self, idx):
        user = torch.tensor(self.user_idx[idx], dtype=torch.long)
        item = torch.tensor(self.item_idx[idx], dtype=torch.long)
        weight = torch.tensor(self.weights[idx], dtype=torch.float32)

        user_feats = torch.tensor(self.user_features[idx], dtype=torch.long)
        item_feats = torch.tensor(self.item_features[idx], dtype=torch.long)

        return user, item, user_feats, item_feats, weight

In [ ]:

class TwoTowerModel(nn.Module):
    def __init__(self, num_users, num_items,
                 user_cat_sizes,
                 item_cat_sizes,
                 emb_dim=64, hidden_dims=[128, 64], dropout=0.2):
        super().__init__()
        self.user_id_emb = nn.Embedding(num_users, emb_dim)
        self.item_id_emb = nn.Embedding(num_items, emb_dim)

        self.user_cat_embs = nn.ModuleList([nn.Embedding(size, emb_dim) for size in user_cat_sizes])
        self.item_cat_embs = nn.ModuleList([nn.Embedding(size, emb_dim) for size in item_cat_sizes])

        user_input_dim = emb_dim * (1 + len(user_cat_sizes))
        item_input_dim = emb_dim * (1 + len(item_cat_sizes))

        user_layers = []
        in_dim = user_input_dim
        for hdim in hidden_dims:
            user_layers.append(nn.Linear(in_dim, hdim))
            user_layers.append(nn.BatchNorm1d(hdim))
            user_layers.append(nn.ReLU())
            user_layers.append(nn.Dropout(dropout))
            in_dim = hdim
        self.user_mlp = nn.Sequential(*user_layers)

        item_layers = []
        in_dim = item_input_dim
        for hdim in hidden_dims:
            item_layers.append(nn.Linear(in_dim, hdim))
            item_layers.append(nn.BatchNorm1d(hdim))
            item_layers.append(nn.ReLU())
            item_layers.append(nn.Dropout(dropout))
            in_dim = hdim
        self.item_mlp = nn.Sequential(*item_layers)

        self.user_proj = nn.Linear(hidden_dims[-1], emb_dim)
        self.item_proj = nn.Linear(hidden_dims[-1], emb_dim)

    def forward(self, user_ids, item_ids, user_cat_feats, item_cat_feats):
        # user_cat_feats: список тензоров [batch_size] или один тензор [batch_size, num_cat_feats]
        # но ожидаем, что приходит тензор формы [batch_size, num_features] – тогда нужно разбить
        # Для удобства будем передавать user_cat_feats как список тензоров (каждый размер [batch_size])
        user_emb = self.user_id_emb(user_ids)  # [batch, emb_dim]
        user_emb_list = [user_emb]
        for i, emb_layer in enumerate(self.user_cat_embs):
            user_emb_list.append(emb_layer(user_cat_feats[i]))  # [batch, emb_dim]
        user_vec = torch.cat(user_emb_list, dim=1)  # [batch, user_input_dim]
        user_vec = self.user_mlp(user_vec)          # [batch, hidden_dims[-1]]
        user_vec = self.user_proj(user_vec)         # [batch, emb_dim]

        item_emb = self.item_id_emb(item_ids)       # [batch, emb_dim]
        item_emb_list = [item_emb]
        for i, emb_layer in enumerate(self.item_cat_embs):
            item_emb_list.append(emb_layer(item_cat_feats[i]))
        item_vec = torch.cat(item_emb_list, dim=1)  # [batch, item_input_dim]
        item_vec = self.item_mlp(item_vec)          # [batch, hidden_dims[-1]]
        item_vec = self.item_proj(item_vec)         # [batch, emb_dim]

        # Скалярное произведение (оценка сходства)
        scores = (user_vec * item_vec).sum(dim=1)
        return scores

    def get_user_embedding(self, user_ids, user_cat_feats):
        """Получение эмбеддинга пользователя для инференса (без товарной части)"""
        user_emb = self.user_id_emb(user_ids)
        user_emb_list = [user_emb]
        for i, emb_layer in enumerate(self.user_cat_embs):
            user_emb_list.append(emb_layer(user_cat_feats[i]))
        user_vec = torch.cat(user_emb_list, dim=1)
        user_vec = self.user_mlp(user_vec)
        user_vec = self.user_proj(user_vec)
        return user_vec

    def get_item_embedding(self, item_ids, item_cat_feats):
        """Получение эмбеддинга товара для инференса"""
        item_emb = self.item_id_emb(item_ids)
        item_emb_list = [item_emb]
        for i, emb_layer in enumerate(self.item_cat_embs):
            item_emb_list.append(emb_layer(item_cat_feats[i]))
        item_vec = torch.cat(item_emb_list, dim=1)
        item_vec = self.item_mlp(item_vec)
        item_vec = self.item_proj(item_vec)
        return item_vec

In [12]:
def train_one_epoch(model, loader, optimizer, device,all_item_features,  num_negs=5, log_interval=50):
    model.train()
    total_loss = 0
    pbar = tqdm(loader, desc="Training", unit="batch", leave=False, position=0)

    for batch_idx, (user_batch, pos_item_batch, user_feats, pos_item_feats, weights) in enumerate(pbar):
        user_batch = user_batch.to(device)
        pos_item_batch = pos_item_batch.to(device)
        user_feats = user_feats.to(device)
        pos_item_feats = pos_item_feats.to(device)
        weights = weights.to(device)
        batch_size = pos_item_batch.shape[0]

        # Генерация отрицательных товаров (случайных)
        neg_item_batch = torch.randint(0, model.item_id_emb.num_embeddings,
                                       (batch_size, num_negs), device=device)
        pos_expanded = pos_item_batch.unsqueeze(1)  # (batch_size, 1)
        mask = (neg_item_batch == pos_expanded)
        while mask.any():
            num_replace = mask.sum().item()
            neg_item_batch[mask] = torch.randint(0, model.item_id_emb.num_embeddings,
                                                 (num_replace,), device=device)
            mask = (neg_item_batch == pos_expanded)

        neg_item_feats = all_item_features[neg_item_batch]  # (batch_size, num_negs, feat_dim)

        user_batch_multi = user_batch.repeat_interleave(num_negs)  # (batch_size * num_negs)
        user_feats_multi = user_feats.repeat_interleave(num_negs, dim=0)  # (batch_size*num_negs, feat_dim)
        neg_item_batch_flat = neg_item_batch.flatten()  # (batch_size * num_negs)
        neg_item_feats_flat = neg_item_feats.view(-1, neg_item_feats.shape[-1])  # (batch_size*num_negs, feat_dim)

        # Скоринг для позитивных (один на пользователя)
        user_feats_list = [user_feats[:, i] for i in range(user_feats.shape[1])]
        pos_item_feats_list = [pos_item_feats[:, i] for i in range(pos_item_feats.shape[1])]
        pos_scores = model(user_batch, pos_item_batch, user_feats_list, pos_item_feats_list)  # [batch_size]

        # Скоринг для негативов (все num_negs на пользователя)
        user_feats_list_multi = [user_feats_multi[:, i] for i in range(user_feats_multi.shape[1])]
        neg_item_feats_list = [neg_item_feats_flat[:, i] for i in range(neg_item_feats_flat.shape[1])]
        neg_scores = model(user_batch_multi, neg_item_batch_flat,
                           user_feats_list_multi, neg_item_feats_list)  # [batch_size * num_negs]
        neg_scores = neg_scores.view(batch_size, num_negs)  # [batch_size, num_negs]

        log_sigmoid = torch.log(torch.sigmoid(pos_scores.unsqueeze(1) - neg_scores) + 1e-8)  # [batch_size, num_negs]
        # Взвешенное среднее: вес применяется ко всей строке (ко всем негативам этого пользователя)
        # Умножаем weights[:, None] на log_sigmoid, усредняем по всем элементам
        loss = - (weights.unsqueeze(1) * log_sigmoid).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if (batch_idx + 1) % log_interval == 0 or batch_idx == len(loader) - 1:
            avg_loss = total_loss / (batch_idx + 1)
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'avg': f'{avg_loss:.4f}'})

    return total_loss / len(loader)

In [13]:
def create_tow_tower_dataset(transactions, articles, customers):
    transactions = transactions[transactions['t_dat'] > '2020-08-21']
    transactions = transactions.groupby(['customer_id', 'article_id']).agg(
        {'price': 'sum', 't_dat': 'count'}).reset_index()
    transactions = transactions[['customer_id', 'article_id', 'price']]

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    user_cat_cols = ['age_group', 'club_member_status', 'fashion_news_frequency']

    user_encoders = {}
    user_cat_sizes = {}

    for col in user_cat_cols:
        le = LabelEncoder()
        customers[col] = customers[col].fillna('Unknown').astype(str)
        customers[col] = le.fit_transform(customers[col])
        user_encoders[col] = le
        user_cat_sizes[col] = len(le.classes_)

    user_features_dict = {}
    for uid, group in customers.groupby('customer_id'):
        user_features_dict[uid] = [group[col].iloc[0] for col in user_cat_cols]

    item_cat_cols = ['product_group_name', 'colour_group_name']

    item_encoders = {}
    item_cat_sizes = {}

    for col in item_cat_cols:
        le = LabelEncoder()
        articles[col] = articles[col].fillna('Unknown').astype(str)
        articles[col] = le.fit_transform(articles[col])
        item_encoders[col] = le
        item_cat_sizes[col] = len(le.classes_)

    item_features_dict = {}
    for aid, group in articles.groupby('article_id'):
        item_features_dict[aid] = [group[col].iloc[0] for col in item_cat_cols]

    all_user_ids = customers['customer_id'].unique()
    all_article_ids = articles['article_id'].unique()

    user_id_to_idx = {uid: i for i, uid in enumerate(all_user_ids)}
    item_id_to_idx = {aid: i for i, aid in enumerate(all_article_ids)}

    num_users = len(user_id_to_idx)
    num_items = len(item_id_to_idx)
    transactions['weight'] = transactions['price']

    # Преобразуем ID в индексы
    transactions['user_idx'] = transactions['customer_id'].map(user_id_to_idx)
    transactions['item_idx'] = transactions['article_id'].map(item_id_to_idx)

    # Отбросим строки, где ID не найден (на случай несоответствия)
    transactions = transactions.dropna(subset=['user_idx', 'item_idx'])
    transactions['user_idx'] = transactions['user_idx'].astype(int)
    transactions['item_idx'] = transactions['item_idx'].astype(int)

    dataset = HMDataset(transactions, user_features_dict, item_features_dict,
              user_cat_cols, item_cat_cols, user_id_to_idx, item_id_to_idx)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

    user_cat_sizes_list = [user_cat_sizes[col] for col in user_cat_cols]
    item_cat_sizes_list = [item_cat_sizes[col] for col in item_cat_cols]

    all_item_features = torch.zeros((num_items, len(item_cat_sizes_list)), dtype=torch.long)
    for aid, idx in item_id_to_idx.items():
        feats = item_features_dict.get(aid, [0] * len(item_cat_sizes_list))
        all_item_features[idx] = torch.tensor(feats,device=device)
    all_item_features = all_item_features.to(device)
    user_features_dict_idx = {}
    for uid, feats in user_features_dict.items():
        if uid in user_id_to_idx:
            user_features_dict_idx[user_id_to_idx[uid]] = torch.tensor(feats)
            
    return dataset, train_dataset, val_dataset, all_item_features, user_cat_sizes_list, item_cat_sizes_list, user_features_dict_idx, num_users, num_items

In [14]:

def fast_evaluate_map_at_k(model, val_loader, device, num_items, all_item_features, user_features_dict,
                           k=12):
    all_item_features = all_item_features.to(device)
    model.eval()

    user_to_pos = {}
    for batch in val_loader:
        user_ids, item_ids, _, _, _ = batch
        for u, i in zip(user_ids.cpu().numpy(), item_ids.cpu().numpy()):
            user_to_pos.setdefault(u, set()).add(i)
    users = list(user_to_pos.keys())

    item_indices = np.arange(num_items)

    with torch.no_grad():
        all_item_embs = []
        for start in range(0, len(item_indices), 512):
            batch_ids = torch.tensor(item_indices[start:start + 512], device=device)
            batch_feats = all_item_features[batch_ids].to(device)
            feats_list = [batch_feats[:, i] for i in range(batch_feats.shape[1])]
            embs = model.get_item_embedding(batch_ids, feats_list).cpu()
            all_item_embs.append(embs)
        all_item_embs = torch.cat(all_item_embs, dim=0)  # (n_items_selected, emb_dim)

    user_embs = {}
    for user in tqdm(users, desc="Evaluating MAP (fast)", leave=False, position=0):
        u_tensor = torch.tensor([user], device=device)
        u_feats = user_features_dict[user].to(device)  # тензор (len(user_cat_sizes),)
        feats_list = [u_feats[i].unsqueeze(0) for i in range(u_feats.shape[0])]
        with torch.no_grad():
            emb = model.get_user_embedding(u_tensor, feats_list).cpu()
        user_embs[user] = emb  # (1, emb_dim)

    ap_scores = []
    for user in users:
        pos_set = user_to_pos[user]
        if not pos_set:
            continue
        pos_set = {p for p in pos_set if p in item_indices}
        if not pos_set:
            continue
        user_vec = user_embs[user][0]  # (emb_dim,)
        scores = torch.matmul(all_item_embs, user_vec)  # (n_items_selected,)
        top_k_indices = torch.topk(scores, k=k).indices.cpu().numpy()
        top_items = [item_indices[idx] for idx in top_k_indices]
        hits = 0
        prec_sum = 0.0
        for rank, item in enumerate(top_items, 1):
            if item in pos_set:
                hits += 1
                prec_sum += hits / rank
        ap = prec_sum / min(k, len(pos_set))
        ap_scores.append(ap)
    return np.mean(ap_scores) if ap_scores else 0.0

In [15]:
 if CFG.optimize_two_tower:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device [{device}]")
    dataset, train_dataset, val_dataset, all_item_features, user_cat_sizes_list, item_cat_sizes_list, user_features_dict_idx, num_users, num_items = create_tow_tower_dataset(transactions, articles, customers)
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=42),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=3)
    )

    def objective(trial):
        emb_dim = trial.suggest_categorical('emb_dim', [32, 64, 128])
        lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical('batch_size', [256, 512, 1024])
        dropout = trial.suggest_float('dropout', 0.1, 0.5)
        hidden_dims = trial.suggest_categorical('hidden_dims', [[64, 32], [128, 64], [256, 128]])
        weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
        num_epochs_fast = 5
    
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=1)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=1)
    
        model = TwoTowerModel(
            num_users=num_users,
            num_items=num_items,
            user_cat_sizes=user_cat_sizes_list,
            item_cat_sizes=item_cat_sizes_list,
            emb_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout
        ).to(device)
    
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
        for epoch in range(num_epochs_fast):
            train_loss = train_one_epoch(model, train_loader, optimizer,  device, all_item_features, log_interval=100)
    
        final_map = fast_evaluate_map_at_k(
            model, val_loader, device, num_items,
            all_item_features, user_features_dict_idx,
            k=12)
        return final_map

    study.optimize(objective, n_trials=30, n_jobs=1, show_progress_bar=True, callbacks=[create_save_best_params_callback('best_params_two_tower.json')])

    print("Best trial:")
    best_trial = study.best_trial
    print(f"MAP@12: {best_trial.value}")
    print(f"Params: [{best_trial.params}]")

In [16]:
if CFG.train_tow_tower:
    dataset, train_dataset, val_dataset, all_item_features, user_cat_sizes_list, item_cat_sizes_list, user_features_dict_idx, num_users, num_items = create_tow_tower_dataset(transactions, articles, customers)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    batch_size = 512
    epochs = 15
    with open('best_params_two_tower.json', 'r') as file:
        best_params = json.load(file)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=1)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=1)

    model = TwoTowerModel(
        num_users=num_users,
        num_items=num_items,
        user_cat_sizes=user_cat_sizes_list,
        item_cat_sizes=item_cat_sizes_list,
        emb_dim=best_params['emb_dim'],
        hidden_dims=best_params['hidden_dims'],
        dropout=best_params["dropout"]
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=best_params['lr'], weight_decay=best_params["weight_decay"])
    losses = []
    maps = []
    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, device,all_item_features, log_interval=100)
        print(f'Loss {train_loss}')
        losses.append(train_loss)

    map12 = fast_evaluate_map_at_k(
        model, val_loader, device, num_items,
        all_item_features, user_features_dict_idx,
        k=12)
    print(f"MAP@12 {map12}")
    print(f"Loses {losses}")

### Подбор гиперпараметров и обучение CatBoost

In [17]:
def make_candidate_df(transactions, articles, customers, target_df, val_start_date = '2020-01-01', train=False):
    if train:
        top300 = target_df.groupby('article_id')['customer_id'].nunique().sort_values(ascending=False).head(300).index
    else:
        top300 = transactions.query(f"t_dat >= '{val_start_date}'").groupby('article_id')[
            'customer_id'].nunique().sort_values(ascending=False).head(300).index

    customers_list = target_df['customer_id'].unique()
    # Создаём все комбинации с топ-300 товарами
    df = pd.DataFrame({'customer_id': customers_list})
    df['tmp'] = 0
    tmp2 = pd.DataFrame({'article_id': top300, 'tmp': 0})
    df = pd.merge(df, tmp2, on='tmp', how='outer').drop('tmp', axis=1)

    # Шаг 2: товары с тем же product_code, что уже купленные пользователем
    user_products = transactions[['customer_id', 'article_id']].drop_duplicates()
    user_products = user_products[user_products['customer_id'].isin(customers_list)]
    user_products = pd.merge(user_products, articles[['article_id', 'product_code']], on='article_id', how='left')
    user_products = user_products.drop_duplicates(['customer_id', 'product_code'])[['customer_id', 'product_code']]

    # Все товары, имеющие такие product_code
    articles_with_codes = articles[articles['product_code'].isin(user_products['product_code'].unique())][
        ['article_id', 'product_code']]
    extra = pd.merge(user_products, articles_with_codes, on='product_code', how='outer')[['customer_id', 'article_id']]
    extra = extra.dropna(subset=['customer_id', 'article_id']).astype({'customer_id': 'int32', 'article_id': 'int32'})

    # Объединяем и убираем дубликаты
    df = pd.concat([df, extra]).drop_duplicates(['customer_id', 'article_id']).reset_index(drop=True)
    df = df.merge(customers, on='customer_id', how='left')
    if train:
        # Добавляем метку target (1 если покупка в целевой период)
        target_pairs = target_df[['customer_id', 'article_id']].drop_duplicates()
        target_pairs['target'] = 1
        df = pd.merge(df, target_pairs, on=['customer_id', 'article_id'], how='left')
        df['target'] = df['target'].fillna(0).astype('int8')
    else:
        df['target'] = 0

    return df

def build_full_dataset(transactions, articles, customers, target_df, train=False):
    le = LabelEncoder()
    all_customers = pd.concat([transactions['customer_id'], customers['customer_id']]).unique()
    le.fit(all_customers)
    transactions['customer_id'] = le.transform(transactions['customer_id']).astype('int32')
    customers['customer_id'] = le.transform(customers['customer_id']).astype('int32')
    target_df['customer_id'] = le.transform(target_df['customer_id']).astype('int32')
    print("Формирование пар (customer, article)...")
    df_copy = make_candidate_df(transactions, articles, customers, target_df, train=train)
    for col in ['club_member_status', 'fashion_news_frequency', 'postal_code', 'age_group', 'recency_cat', 'frequency_cat', 'unique_articles_cat']:
        le = LabelEncoder()
        df_copy[col] = le.fit_transform(df_copy[col])
    drop_cols = ['customer_id', 'article_id', 'target']
    if train:
        drop_cols.append('target')
    feature_cols = [c for c in df_copy.columns if c not in drop_cols]
    return df_copy, feature_cols

In [18]:
def mapk(actual, predicted, k=12):
    def apk(a, p, k):
        if len(p) > k:
            p = p[:k]
        score = 0.0
        hits = 0.0
        for i, pred in enumerate(p):
            if pred in a and pred not in p[:i]:
                hits += 1.0
                score += hits / (i + 1.0)
        return score / min(len(a), k) if a else 0.0

    return np.mean([apk(a, p, k) for a, p in zip(actual, predicted)])

In [19]:
if CFG.optimize_catboost:
    n_folds = 5
    val_start_date = '2020-09-16'
    transactions = transactions[transactions['t_dat'] > '2020-08-21']
    target_df = transactions[transactions['t_dat'] >= val_start_date].reset_index(drop=True)
    train_transactions = transactions[transactions['t_dat'] < val_start_date].reset_index(drop=True)
    df, feature_cols = build_full_dataset(train_transactions, articles, customers.copy(), target_df, train=True)
    print("Подбор гиперпараметров CatBoost с кросс-валидацией...")
    unique_customers = df['customer_id'].unique()
    
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=CFG.seed)
    fold_assign = np.zeros(len(df), dtype=int)
    for i, (_, val_idx) in enumerate(kf.split(unique_customers)):
        val_cust = unique_customers[val_idx]
        fold_assign[df['customer_id'].isin(val_cust)] = i
    
    def objective(trial):
        learning_rate = trial.suggest_float('learning_rate', 0.005, 0.1, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 10)
        l2_leaf_reg = trial.suggest_float('l2_leaf_reg', 1e-5, 10.0, log=True)
        border_count = trial.suggest_int('border_count', 32, 255)
        scale_pos_weight = trial.suggest_float('scale_pos_weight', 10, 200, log=True)
        iterations = trial.suggest_int('iterations', 500, 2000)  # ограничим для ускорения
        early_stopping_rounds = 100
        
        params = {
            'loss_function': 'Logloss',
            'learning_rate': learning_rate,
            'max_depth': max_depth,
            'l2_leaf_reg': l2_leaf_reg,
            'border_count': border_count,
            'scale_pos_weight': scale_pos_weight,
            'iterations': iterations,
            'early_stopping_rounds': early_stopping_rounds,
            'random_seed': CFG.seed,
            'thread_count': 22,
            'task_type': 'CPU',
            'verbose': True,
            'gpu_cat_features_storage': 'CpuPinnedMemory',
        }
        
        cat_cols = [c for c in feature_cols if c in ['product_code', 'product_type_no', 'graphical_appearance_no',
                                                      'colour_group_code', 'index_group_no', 'section_no',
                                                      'garment_group_no']]
        oof_pred = np.zeros(len(df))
        scores = []
        for fold in range(n_folds):
            tr_idx = fold_assign != fold
            va_idx = fold_assign == fold
            X_train = df.loc[tr_idx, feature_cols]
            y_train = df.loc[tr_idx, 'target']
            X_val = df.loc[va_idx, feature_cols]
            y_val = df.loc[va_idx, 'target']
            
            train_pool = Pool(X_train, y_train, cat_features=cat_cols)
            val_pool = Pool(X_val, y_val, cat_features=cat_cols)
            
            model = CatBoostClassifier(**params)
            model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=True)
            
            val_pred = model.predict_proba(X_val)[:, 1]
            oof_pred[va_idx] = val_pred
            # Считаем MAP@12 на валидации
            val_df = df.loc[va_idx, ['customer_id', 'article_id', 'target']].copy()
            val_df['pred'] = val_pred
            val_group = val_df.sort_values('pred', ascending=False).groupby('customer_id')['article_id'].apply(list).reset_index()
            target_group = target_df.groupby('customer_id')['article_id'].apply(list).reset_index()
            val_group = pd.merge(val_group, target_group, on='customer_id', how='left')
            score = mapk(val_group['article_id_y'], val_group['article_id_x'], k=12)
            scores.append(score)
        avg_score = np.mean(scores)
        return avg_score
    
    
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=CFG.seed))
    study.optimize(objective, n_trials=30, show_progress_bar=True, callbacks=[create_save_best_params_callback('best_params_catboost.json')])
    
    print("Best trial:")
    best_trial = study.best_trial
    print(f"MAP@12: {best_trial.value}")
    print(f'Best params [{best_trial.params}]')

In [20]:
if CFG.train_catboost:
    with open('best_params_catboost.json', 'r') as file:
        best_params = json.load(file)
    best_params.update({
        'loss_function': 'Logloss',
        'random_seed': CFG.seed,
        'thread_count':22,
        'task_type': 'CPU',
        'verbose': True,
        'gpu_cat_features_storage': 'CpuPinnedMemory',
    })

    n_folds = 5
    val_start_date = '2020-09-16'
    transactions = transactions[transactions['t_dat'] > '2020-08-21']
    target_df = transactions[transactions['t_dat'] >= val_start_date].reset_index(drop=True)
    train_transactions = transactions[transactions['t_dat'] < val_start_date].reset_index(drop=True)
    df, feature_cols = build_full_dataset(train_transactions, articles, customers.copy(), target_df, train=True)
    print("Обучение CatBoost с кросс-валидацией...")
    unique_customers = df['customer_id'].unique()
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=CFG.seed)
    fold_assign = np.zeros(len(df), dtype=int)
    for i, (_, val_idx) in enumerate(kf.split(unique_customers)):
        val_cust = unique_customers[val_idx]
        fold_assign[df['customer_id'].isin(val_cust)] = i

    cat_cols = [c for c in feature_cols if c in ['product_code', 'product_type_no', 'graphical_appearance_no',
                                                      'colour_group_code', 'index_group_no', 'section_no',
                                                      'garment_group_no']]
    oof_pred = np.zeros(len(df))
    scores = []
    for fold in range(n_folds):
        tr_idx = fold_assign != fold
        va_idx = fold_assign == fold
        X_train = df.loc[tr_idx, feature_cols]
        y_train = df.loc[tr_idx, 'target']
        X_val = df.loc[va_idx, feature_cols]
        y_val = df.loc[va_idx, 'target']
        
        train_pool = Pool(X_train, y_train, cat_features=cat_cols)
        val_pool = Pool(X_val, y_val, cat_features=cat_cols)
        
        model = CatBoostClassifier(**best_params)
        model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=True)
        
        val_pred = model.predict_proba(X_val)[:, 1]
        oof_pred[va_idx] = val_pred
        # Считаем MAP@12 на валидации
        val_df = df.loc[va_idx, ['customer_id', 'article_id', 'target']].copy()
        val_df['pred'] = val_pred
        val_group = val_df.sort_values('pred', ascending=False).groupby('customer_id')['article_id'].apply(list).reset_index()
        target_group = target_df.groupby('customer_id')['article_id'].apply(list).reset_index()
        val_group = pd.merge(val_group, target_group, on='customer_id', how='left')
        score = mapk(val_group['article_id_y'], val_group['article_id_x'], k=12)
        scores.append(score)
        print(f"Fold {fold} MAP@12 = {score:.6f}")
    print(f"Средний MAP@12 по фолдам: {np.mean(scores):.6f}")

### Сравнение моделей. 

In [23]:
import numpy as np
from sklearn.model_selection import KFold
from tqdm import tqdm
import torch
from torch.utils.data import DataLoader
import json
import warnings
warnings.filterwarnings('ignore')
from lightfm import LightFM
from lightfm.data import Dataset
import pandas as pd

with open('best_params_lightfm.json', 'r') as file:
    lightfm_params = json.load(file)

with open('best_params_two_tower.json', 'r') as file:
    twotower_params = json.load(file)

with open('best_params_catboost.json', 'r') as file:
    catboost_params = json.load(file)

catboost_params.update({
        'loss_function': 'Logloss',
        'thread_count':22,
        'task_type': 'GPU',
        'gpu_cat_features_storage': 'CpuPinnedMemory',
    })

epochs = 20

def train_and_evaluate_lightfm(train_trans, val_trans, customers, articles, params):
    user_feature_matrix,  item_feature_matrix, _, _, interactions_matrix, weights_matrix = get_lightfm_dataset(train_trans.copy(), customers.copy(), articles.copy())
    _,  _, _, _, val_interactions_matrix, _ = get_lightfm_dataset(val_trans.copy(), customers.copy(), articles.copy())
    train_agg = train_trans.groupby(['customer_id','article_id']).agg({'price':'sum'}).reset_index()
    train_agg['price'] = train_agg['price'].astype('category').cat.codes
    model = LightFM(**params, random_state=CFG.seed)
    model.fit(interactions_matrix, user_features=user_feature_matrix, item_features=item_feature_matrix,epochs=epochs, num_threads=22, verbose=True)

    return precision_at_k(
                model,
                interactions_matrix,
                val_interactions_matrix,
                user_features=user_feature_matrix,
                item_features=item_feature_matrix,
                k=12,
                num_threads=22,
                check_intersections=True
            ).mean()

def train_and_evaluate_twotower(train_trans, val_trans, customers, articles, params):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_dataset, _, _, all_item_features, user_cat_sizes_list, item_cat_sizes_list, user_features_dict_idx, num_users, num_items = create_tow_tower_dataset(train_trans.copy(), articles.copy(), customers.copy())
    val_dataset, _, _, all_item_features, user_cat_sizes_list, item_cat_sizes_list, user_features_dict_idx, num_users, num_items = create_tow_tower_dataset(val_trans.copy(), articles.copy(), customers.copy())
    
    train_loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=params['batch_size'], shuffle=False, num_workers=0)

    model = TwoTowerModel(num_users, num_items, user_cat_sizes_list, item_cat_sizes_list,
                          emb_dim=params['emb_dim'], 
                          hidden_dims=params['hidden_dims'],
                          dropout=params['dropout']).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])

    for epoch in range(epochs):
        train_one_epoch(model, train_loader, optimizer, device, all_item_features, num_negs=5, log_interval=100)

    map12 = fast_evaluate_map_at_k(model, val_loader, device, num_items,
                                   all_item_features, user_features_dict_idx, k=12)
    return map12

def train_and_evaluate_catboost(train_trans, val_trans, customers, articles, params):
    train_trans = train_trans.copy()
    val_trans = val_trans.copy()
    customers = customers.copy()
    articles = articles.copy()
    target_df = val_trans[['customer_id','article_id']].drop_duplicates()
    df, feature_cols = build_full_dataset(train_trans.copy(), articles.copy(), customers.copy(), target_df, train=True)
    cat_cols = [c for c in feature_cols if c in ['product_code', 'product_type_no', 'graphical_appearance_no',
                                                  'colour_group_code', 'index_group_no', 'section_no',
                                                  'garment_group_no']]
    model = CatBoostClassifier(**params, random_seed=CFG.seed, verbose=True)
    train_pool = Pool(df[feature_cols], df['target'], cat_features=cat_cols)
    model.fit(train_pool)
    preds = model.predict_proba(df[feature_cols])[:,1]
    df['pred'] = preds
    user_preds = df.sort_values('pred', ascending=False).groupby('customer_id')['article_id'].apply(list).to_dict()
    real_dict = target_df.groupby('customer_id')['article_id'].apply(list).to_dict()
    map_scores = []
    for uid in real_dict.keys():
        if uid not in user_preds:
            continue
        pred_list = user_preds[uid][:12]
        real_set = set(real_dict[uid])
        hits = 0
        prec_sum = 0.0
        for rank, art in enumerate(pred_list, 1):
            if art in real_set:
                hits += 1
                prec_sum += hits / rank
        ap = prec_sum / min(12, len(real_set))
        map_scores.append(ap)
    return np.mean(map_scores) if map_scores else 0.0

def cross_validate_models(transactions, customers, articles, n_folds=5):
    unique_users = customers['customer_id'].unique()
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=CFG.seed)
    results = {'LightFM': [], 'TwoTower': [], 'CatBoost': []}

    for fold, (train_idx, val_idx) in enumerate(kf.split(unique_users)):
        print(f"=== Fold {fold+1}/{n_folds} ===")
        train_users = set(unique_users[train_idx])
        val_users = set(unique_users[val_idx])

        train_trans = transactions[transactions['customer_id'].isin(train_users)].copy()
        val_trans = transactions[transactions['customer_id'].isin(val_users)].copy()
        
        print("LightFM...")
        lfm = train_and_evaluate_lightfm(train_trans, val_trans, customers, articles, lightfm_params)
        results['LightFM'].append(lfm)
        print(f"MAP@12 = {lfm:.6f}")
        print("TwoTower...")
        tt = train_and_evaluate_twotower(train_trans, val_trans, customers, articles, twotower_params)
        results['TwoTower'].append(tt)
        print(f"MAP@12 = {tt:.6f}")
        print("CatBoost...")
        cb = train_and_evaluate_catboost(train_trans, val_trans, customers, articles, catboost_params)
        results['CatBoost'].append(cb)
        print(f"MAP@12 = {cb:.6f}")

    return results



In [24]:
cv_scores = cross_validate_models(transactions, customers, articles, n_folds=5)


=== Fold 1/5 ===
TwoTower...


MAP@12 = 0.000329
CatBoost...
Формирование пар (customer, article)...


0:	learn: 0.6715551	total: 94.5ms	remaining: 1m 9s
1:	learn: 0.6509171	total: 187ms	remaining: 1m 8s
2:	learn: 0.6311629	total: 281ms	remaining: 1m 8s
3:	learn: 0.6121959	total: 371ms	remaining: 1m 7s
4:	learn: 0.5940966	total: 466ms	remaining: 1m 7s
5:	learn: 0.5768787	total: 546ms	remaining: 1m 6s
6:	learn: 0.5602720	total: 641ms	remaining: 1m 6s
7:	learn: 0.5443135	total: 733ms	remaining: 1m 6s
8:	learn: 0.5291770	total: 806ms	remaining: 1m 4s
9:	learn: 0.5146598	total: 899ms	remaining: 1m 5s
10:	learn: 0.5007104	total: 1s	remaining: 1m 5s
11:	learn: 0.4875361	total: 1.1s	remaining: 1m 6s
12:	learn: 0.4749571	total: 1.2s	remaining: 1m 6s
13:	learn: 0.4631906	total: 1.25s	remaining: 1m 4s
14:	learn: 0.4515981	total: 1.34s	remaining: 1m 4s
15:	learn: 0.4403655	total: 1.44s	remaining: 1m 4s
16:	learn: 0.4298462	total: 1.56s	remaining: 1m 5s
17:	learn: 0.4197300	total: 1.65s	remaining: 1m 5s
18:	learn: 0.4100941	total: 1.74s	remaining: 1m 5s
19:	learn: 0.4008110	total: 1.84s	remaining: 

MAP@12 = 0.000317
CatBoost...
Формирование пар (customer, article)...


0:	learn: 0.6717000	total: 96.5ms	remaining: 1m 10s
1:	learn: 0.6510857	total: 194ms	remaining: 1m 11s
2:	learn: 0.6313015	total: 290ms	remaining: 1m 10s
3:	learn: 0.6123200	total: 387ms	remaining: 1m 10s
4:	learn: 0.5942901	total: 510ms	remaining: 1m 14s
5:	learn: 0.5771065	total: 597ms	remaining: 1m 12s
6:	learn: 0.5605033	total: 701ms	remaining: 1m 12s
7:	learn: 0.5445854	total: 793ms	remaining: 1m 12s
8:	learn: 0.5294005	total: 930ms	remaining: 1m 14s
9:	learn: 0.5148538	total: 1.02s	remaining: 1m 14s
10:	learn: 0.5009563	total: 1.12s	remaining: 1m 13s
11:	learn: 0.4878616	total: 1.22s	remaining: 1m 13s
12:	learn: 0.4752527	total: 1.32s	remaining: 1m 13s
13:	learn: 0.4631735	total: 1.44s	remaining: 1m 14s
14:	learn: 0.4515152	total: 1.53s	remaining: 1m 13s
15:	learn: 0.4405459	total: 1.65s	remaining: 1m 13s
16:	learn: 0.4301066	total: 1.74s	remaining: 1m 13s
17:	learn: 0.4200608	total: 1.85s	remaining: 1m 13s
18:	learn: 0.4104832	total: 1.96s	remaining: 1m 13s
19:	learn: 0.4012881	

KeyboardInterrupt: 

In [ ]:

print("Результаты кросс-валидации")
for name, scores in cv_scores.items():
    arr = np.array(scores)
    print(f"{name}: mean = {arr.mean():.6f}, std = {arr.std(ddof=1):.6f}")
    print(f"scores: {scores}")

def bootstrap_ci(scores_a, scores_b, n=10000):
    np.random.seed(42)
    diffs = []
    for _ in range(n):
        idx = np.random.randint(0, len(scores_a), len(scores_a))
        diffs.append(np.mean(scores_a[idx]) - np.mean(scores_b[idx]))
    low, high = np.percentile(diffs, [2.5, 97.5])
    return low, high

print("Сравнение 95% CI разности средних")
models = list(cv_scores.keys())
for i in range(len(models)):
    for j in range(i+1, len(models)):
        a, b = models[i], models[j]
        low, high = bootstrap_ci(cv_scores[a], cv_scores[b])
        diff_obs = np.mean(cv_scores[a]) - np.mean(cv_scores[b])
        print(f"{a} vs {b}: разность = {diff_obs:.6f}   CI = [{low:.6f}, {high:.6f}]")
        if low > 0:
            print(f"{a} статистически лучше {b}")
        elif high < 0:
            print(f"{b} статистически лучше {a}")
        else:
            print(f"значимого различия нет")